In [1]:
import sys 
sys.path.append('C:/Users/DATA/Documents/datos/01_script/inicio/funciones')
from funciones import *
from funciones_spark import *
from variables_inicio import *
from utils_sql import *

# spark = SparkSession.builder \
#     .appName("SparkExample") \
#     .master("local[*]") \
#     .config('spark.driver.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.extraClassPath', 'C:/spark/jars/mssql-jdbc-13.2.1.jre11.jar') \
#     .config('spark.executor.memory', '8g') \
#     .config('spark.driver.memory', '8g') \
#     .getOrCreate()

In [5]:
exec_query_sql(server_zeus, "MAEBA", user_zeus, pwd_zeus, "ADM_OBJ_TG.spFunnelDinersTc", "SP funnel diners_tc Zeus")

SP funnel diners_tc Zeus | realizado | duración: 19.96 seg


In [ ]:
overwrite_table_SQL(spark,df_prueba_1,f'borrar_TARGET_202604_01',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_2,f'borrar_TARGET_202604_02',server_kishin,user_kishin,pwd_kishin,'DANTALION')
overwrite_table_SQL(spark,df_prueba_ch,f'borrar_TARGET_202604_ch',server_kishin,user_kishin,pwd_kishin,'DANTALION')


In [2]:
from sqlalchemy import create_engine


engine_mysql = create_engine(
    f"mysql+pymysql://{user_valentina}:{pwd_valentina}@{server_valentina}:{port_mysql}/{db_valentina}"
)

query = """
SELECT  NUMERO_DOCUMENTO,cl_telf1, cl_telf2, cl_telf3, cl_telf4, cl_telf5, cl_telf6, cl_telf7, cl_telf8, cl_telf9, cl_telf10, cl_movil, cl_celular, cl_telefono FROM crm_target.alfin_clientes
WHERE cl_base = 'mayo 2026'
and cl_estado=1
"""

df_dni = pd.read_sql(query, engine_mysql)

In [3]:
cols_tel = [
    'cl_telf1','cl_telf2','cl_telf3','cl_telf4','cl_telf5',
    'cl_telf6','cl_telf7','cl_telf8','cl_telf9','cl_telf10',
    'cl_movil','cl_celular','cl_telefono'
]

df_long = df_dni.melt(
    id_vars='NUMERO_DOCUMENTO',
    value_vars=cols_tel,
    var_name='tipo_telf',
    value_name='TELEFONO'
)

In [4]:
df_long['TELEFONO'] = (
    df_long['TELEFONO']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)
df_long = df_long[
    (df_long['TELEFONO'].notna()) &
    (df_long['TELEFONO'] != '') &
    (df_long['TELEFONO'].str.len() == 9) &
    (df_long['TELEFONO'].str.startswith('9'))
]

In [5]:
# filename='RetiroDeGestion_BlackList.csv'
filename='RetiroDeGestion_Telefonos.csv'

filePath = os.path.join(ruta_csv, filename)

df_list = pd.read_csv(filePath)
print(f"df_dni filas: {df_long.shape[0]}")
print(f"df_list filas: {df_list.shape[0]}")

df_dni filas: 202629
df_list filas: 1267


In [ ]:
df_dni filas: 128183
df_list filas: 12546

In [6]:
df_list['TELEFONO'] = (
    df_list['TELEFONO']
    .fillna(0)            
    .astype('int64')        
    .astype(str)              
)

In [7]:
print(df_list.columns)
print(df_long.columns)

Index(['TELEFONO'], dtype='object')
Index(['NUMERO_DOCUMENTO', 'tipo_telf', 'TELEFONO'], dtype='object')


In [8]:
df_list = df_list.merge(
    df_long,
    on="TELEFONO",
    how="inner"
)

print(f"df_list filas: {df_list.shape[0]}")


df_list filas: 1


In [ ]:
df_list filas: 2036

In [9]:
df_list['retiro']='Retirar Telef'
df_list=df_list[['NUMERO_DOCUMENTO','retiro']]
df_list.count()

NUMERO_DOCUMENTO    1
retiro              1
dtype: int64

In [10]:
df_list.head()

,NUMERO_DOCUMENTO,retiro
0,40503275,Retirar Telef


In [11]:
update_mysql_en_bloques(
    df=df_list,
    tabla="alfin_clientes",
    periodo="mayo 2026",
    col_llave_mysql="NUMERO_DOCUMENTO",
    col_valor_mysql="estado",
    col_llave_df="NUMERO_DOCUMENTO",
    col_valor_df="retiro",
    host=server_valentina,
    user=user_valentina,
    password=pwd_valentina,
    database=db_valentina,
    port=port_mysql,
    batch_size=2000,
    validar_sin_grabar=False
)


Total registros a procesar: 1
Lote 0 - 1 actualizado | filas afectadas: 1
Proceso terminado. Total filas afectadas: 1
